In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
#from NeuralMF import NeuralMF
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import pytorch_lightning as pl
import optuna
from sklearn.preprocessing import StandardScaler
from optuna.integration import PyTorchLightningPruningCallback
import gc  # 가비지 컬렉션
from sklearn.metrics import mean_squared_error, mean_absolute_error


In [2]:
!pip install pytorch_lightning
!pip install optuna
!pip install optuna-integration[pytorch_lightning]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.0/823.0 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.9/960.9 kB 55.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitli

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# movie_ratings = pd.read_csv('./dataset/ratings.csv')
# movie_ratings_small = pd.read_csv('./dataset/ratings_small.csv')
# movies_metadata = pd.read_csv('./dataset/movies_metadata.csv')

movie_ratings = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movie_dataset/ratings.csv')
movie_ratings_small = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movie_dataset/ratings_small.csv')
movies_metadata = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movie_dataset/movies_metadata.csv')


<ipython-input-5-80c1eb9504cb>:7: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_metadata = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movie_dataset/movies_metadata.csv')


In [6]:
movie_ratings.isnull().sum()

,0
userId,0
movieId,0
rating,0
timestamp,0


In [7]:
movies_metadata.drop_duplicates(subset='id',keep='first', inplace=True)

In [8]:
movies_metadata = movies_metadata[movies_metadata['id'].str.isdigit()]
movies_metadata['id'] = movies_metadata['id'].astype('int64')

<ipython-input-8-7420a33fdfa0>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_metadata['id'] = movies_metadata['id'].astype('int64')


In [9]:
movies_metadata = movies_metadata.merge(movie_ratings_small, left_on='id', right_on='movieId', how='left')

In [10]:
movies_metadata.dropna(subset='userId', inplace= True)
movies_metadata.drop(columns=['movieId'], inplace=True)

In [11]:
movies_metadata.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count', 'userId', 'rating', 'timestamp'],
      dtype='object')

In [12]:
movies_metadata['genres']

,genres
5,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam..."
6,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam..."
7,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam..."
8,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam..."
9,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam..."
...,...
87527,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ..."
87528,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ..."
87529,"[{'id': 10749, 'name': 'Romance'}, {'id': 18, ..."
87532,"[{'id': 35, 'name': 'Comedy'}, {'id': 10749, '..."


In [13]:
import json
def extract_genre_names(metadata):
    # 각 리스트에서 'name' 키 값을 추출
    corrected_json_string = metadata.replace("'", '"')
    json_data = json.loads(corrected_json_string)
    a  = [genre['name'] for genre in json_data if 'name' in genre]
    return a

movies_metadata['genres'] = movies_metadata['genres'].apply(lambda x: extract_genre_names(x))


In [14]:
users_stats = movies_metadata.groupby('userId')['rating'].agg(['mean','std','count']).reset_index()
users_stats.columns = ['userId','user_mean_rating','user_rating_std','user_review_count']
movies_metadata = movies_metadata.merge(users_stats, on='userId', how='left')

In [15]:
movies_metadata['release_year'] = pd.to_datetime(movies_metadata['release_date']).dt.year

movie_stats = movies_metadata.groupby('id')['rating'].agg(['mean', 'count']).reset_index()
movie_stats.columns = ['id', 'movie_mean_rating', 'movie_review_count']

movies_metadata = movies_metadata.merge(movie_stats, on='id', how='left')


In [16]:
movies_metadata.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count', 'userId', 'rating', 'timestamp',
       'user_mean_rating', 'user_rating_std', 'user_review_count',
       'release_year', 'movie_mean_rating', 'movie_review_count'],
      dtype='object')

In [17]:
unique_genres = sorted(set(genre for genres in movies_metadata['genres'] for genre in genres))
genre_to_idx = {genre: idx for idx, genre in enumerate(unique_genres)}


# 장르를 숫자로 변환
movies_metadata['genre_ids'] = movies_metadata['genres'].apply(lambda x: [genre_to_idx[genre] for genre in x])

In [18]:
genre_ratings = movies_metadata.explode('genre_ids').groupby(['userId','genre_ids'])['rating'].mean().reset_index()
genre_ratings.columns = ['userId','genre_ids', 'user_preference']


In [19]:
movies_metadata_exploded = movies_metadata.explode('genre_ids')
movies_metadata_exploded.fillna({'genre_ids': 0}, inplace= True)



<ipython-input-19-2d472db91df7>:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  movies_metadata_exploded.fillna({'genre_ids': 0}, inplace= True)


In [20]:
movies_metadata_exploded = movies_metadata_exploded.merge(genre_ratings, on=['userId','genre_ids'], how='left')

In [21]:
movies_metadata_exploded.isnull().sum()

,0
adult,0
belongs_to_collection,80502
budget,0
genres,0
homepage,81995
id,0
imdb_id,0
original_language,0
original_title,0
overview,150


In [22]:
movies_metadata_exploded[[ 'user_mean_rating', 'user_rating_std']].isnull().sum()


,0
user_mean_rating,0
user_rating_std,0


In [23]:
numeric_features_cols = ['user_mean_rating', 'user_rating_std', 'user_review_count',
       'release_year', 'movie_mean_rating', 'movie_review_count', 'genre_ids',
       'user_preference']
movies_metadata_exploded.fillna({'release_year':0}, inplace=True)
movies_metadata_exploded.fillna({'user_preference':0}, inplace=True)
movies_metadata_exploded.fillna({'user_rating_std':0}, inplace= True)

In [24]:
numeric_features_cols = ['user_mean_rating', 'user_rating_std', 'user_review_count',
       'release_year', 'movie_mean_rating', 'movie_review_count', 'genre_ids',
       'user_preference']

scaler = StandardScaler()
movies_metadata_exploded[numeric_features_cols] = scaler.fit_transform(movies_metadata_exploded[numeric_features_cols])

In [25]:
target = ['user_mean_rating', 'user_rating_std', 'user_review_count',
       'release_year', 'movie_mean_rating', 'movie_review_count', 'genre_ids',
       'user_preference','id','userId','rating']
movies_metadata_exploded[target]

,user_mean_rating,user_rating_std,user_review_count,release_year,movie_mean_rating,movie_review_count,genre_ids,user_preference,id,userId,rating
0,0.207586,-0.354892,0.669992,0.217014,0.049479,-0.826605,-1.405291,0.245843,949,23.0,3.5
1,0.207586,-0.354892,0.669992,0.217014,0.049479,-0.826605,-0.686947,0.574224,949,23.0,3.5
2,0.207586,-0.354892,0.669992,0.217014,0.049479,-0.826605,-0.327775,0.149901,949,23.0,3.5
3,0.207586,-0.354892,0.669992,0.217014,0.049479,-0.826605,1.647671,0.354781,949,23.0,3.5
4,0.949350,-0.427886,0.654369,0.217014,0.049479,-0.826605,-1.405291,0.721596,949,102.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...
110073,0.733337,-1.364769,-0.840205,0.659329,0.817022,-1.027519,-0.866533,0.938690,98604,352.0,4.0
110074,0.733337,-1.364769,-0.840205,0.659329,0.817022,-1.027519,1.108913,0.516978,98604,352.0,4.0
110075,0.320610,0.231559,-0.199674,-2.254746,2.706358,-1.027519,0.031397,1.165766,49280,187.0,5.0
110076,0.320610,0.231559,-0.199674,-2.254746,2.706358,-1.027519,-1.405291,0.268912,49280,187.0,5.0


In [26]:
movies_metadata_exploded[target].isnull().sum()

,0
user_mean_rating,0
user_rating_std,0
user_review_count,0
release_year,0
movie_mean_rating,0
movie_review_count,0
genre_ids,0
user_preference,0
id,0
userId,0


In [27]:
# Neural MF Model
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl

class NeuralMF(pl.LightningModule):
    def __init__(self, num_users, num_items, num_numeric_features, latent_dim, dropout, learning_rate=0.001):
        super(NeuralMF, self).__init__()

        # Embeddings
        self.user_embedding = nn.Embedding(num_users, latent_dim, max_norm=1.0)
        self.item_embedding = nn.Embedding(num_items, latent_dim, max_norm=1.0)
        self.numeric_fc = nn.Linear(num_numeric_features, latent_dim)
        # Neural network layers
        self.gmf_fc = nn.Linear(latent_dim, latent_dim)
        # MLP (Multi-Layer Perceptron)
        self.mlp_fc1 = nn.Linear(latent_dim * 3, 128)
        self.mlp_fc2 = nn.Linear(128, 64)
        self.final_fc = nn.Linear(latent_dim + 64, 1)  # GMF(latent_dim) + MLP(64)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()
        self.learning_rate = learning_rate
        self.criterion = nn.MSELoss()

    def forward(self, user, item, numeric_features):
        user_embed = self.user_embedding(user)
        item_embed = self.item_embedding(item)
        numeric_emb = self.numeric_fc(numeric_features)
        gmf = user_embed * item_embed  # 원소별 곱
        gmf = self.gmf_fc(gmf)  # (batch_size, latent_dim)
        # Concatenate user and item embeddings
        mlp = torch.cat([user_embed, item_embed, numeric_emb], dim=1)
        mlp = self.activation(self.mlp_fc1(mlp))
        mlp = self.dropout(mlp)
        mlp = self.activation(self.mlp_fc2(mlp))  # (batch_size, 64)

        x = torch.cat([gmf, mlp], dim=1)  # (batch_size, latent_dim + 64)
        x = self.final_fc(x)  # (batch_size, 1)
        return (torch.sigmoid(x) * 5).squeeze()

    def training_step(self, batch, batch_idx):
        user_ids, item_ids, numeric_features, ratings = batch
        predicted_ratings = self.forward(user_ids, item_ids, numeric_features)
        train_loss = self.criterion(predicted_ratings, ratings)
        self.log("train_loss", train_loss, prog_bar=True, on_epoch=True, on_step=True)
        return train_loss

    def validation_step(self, batch, batch_idx):
        user_ids, item_ids, numeric_features, ratings = batch
        predicted_ratings = self.forward(user_ids, item_ids, numeric_features)
        loss = self.criterion(predicted_ratings, ratings)
        self.log("validation_loss", loss, prog_bar=True, on_epoch=True, on_step = True)  # 🔥 손실 로깅 추가
        return loss

    def predict_step(self, batch, batch_idx):
        """Lightning에서 `trainer.predict()`를 호출할 때 사용"""
        user, item, numeric_features, ratings = batch  # 배치에서 올바른 입력 추출
        prediction = self.forward(user, item, numeric_features)
        return prediction, ratings

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.learning_rate,  weight_decay=1e-3)

In [30]:
    def prepare(data, is_train, user_map=None, item_map=None):
    # 전체 데이터에서 사용자와 아이템에 대한 고유 인덱스를 얻기 위한 처리
      if user_map is None or item_map is None:
        # train_data와 val_data를 합쳐서 고유한 user_idx, item_idx 매핑을 만듦
        full_data = pd.concat([train_data[['userId', 'id']], val_data[['userId', 'id']]])
        full_data['user_idx'] = full_data['userId'].astype('category').cat.codes
        full_data['item_idx'] = full_data['id'].astype('category').cat.codes
        user_map = dict(zip(full_data['userId'].unique(), full_data['user_idx'].unique()))
        item_map = dict(zip(full_data['id'].unique(), full_data['item_idx'].unique()))

      # 사용자와 아이템을 해당 인덱스로 변환
      data['user_idx'] = data['userId'].map(user_map)
      data['item_idx'] = data['id'].map(item_map)

      user_ids = torch.tensor(data['user_idx'].values, dtype=torch.long)
      item_ids = torch.tensor(data['item_idx'].values, dtype=torch.long)
      ratings = torch.tensor(data['rating'].values, dtype=torch.float32)
      numeric_features = torch.tensor(data[numeric_features_cols].values, dtype=torch.float32)

      num_users = len(user_map)  # 전체 사용자 수
      num_items = len(item_map)  # 전체 아이템 수

      dataset = TensorDataset(user_ids, item_ids, numeric_features, ratings)
      loader = DataLoader(dataset, batch_size=batch_size, shuffle=is_train, num_workers=num_workers)
      return loader, num_users, num_items, numeric_features.shape[1], user_map, item_map


In [28]:
from optuna.integration import PyTorchLightningPruningCallback
data = movies_metadata_exploded[target]
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)
small_train_data = train_data.sample(frac=0.2, random_state=42)
val_data, test_data = train_test_split(val_data, test_size=0.5, random_state=42)  # 검증 & 테스트 분리

def objective(trial):

      # 'latent_dim': 53,
    # 'batch_size': 16,
    # 'learning_rate': 0.0004096453948261441,
    # 'epochs': 11,
    # 'num_workers': 1,
    # 'dropout': 0.24716962709299214
    latent_dim = trial.suggest_int("latent_dim", 50, 128)  # 8~64 사이 정수
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])  # 16, 32, 64 중 선택
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-3, log=True)  # log=True 사용
    epochs = trial.suggest_int("epochs", 10, 50)  # 10~50 사이 정수
    num_workers = trial.suggest_int("num_workers", 0, 4)  # 🔥 num_workers도 최적화 가능!
    dropout = trial.suggest_float("dropout", 0.2, 0.5)  # 드롭아웃 비율


    # def prepare(data, is_train):
    #   data['user_idx'] = data['userId'].astype('category').cat.codes
    #   data['item_idx'] = data['id'].astype('category').cat.codes
    #   user_ids = torch.tensor(data['user_idx'].values, dtype=torch.long)
    #   item_ids = torch.tensor(data['item_idx'].values, dtype=torch.long)
    #   ratings = torch.tensor(data['rating'].values, dtype=torch.float32)
    #   num_users = data['user_idx'].nunique()
    #   num_items = data['item_idx'].nunique()
    #   numeric_features = torch.tensor(data[numeric_features_cols].values, dtype=torch.float32)
    #   dataset = TensorDataset(user_ids, item_ids, numeric_features, ratings)
    #   loader = DataLoader(dataset, batch_size=batch_size, shuffle= is_train, num_workers= num_workers)
    #   return loader, num_users, num_items, numeric_features.shape[1]

    # train_loader, num_users, num_items, num_numeric = prepare(small_train_data, True)
    # val_loader, _, _, _ = prepare(val_data, False)

    def prepare(data, is_train, user_map=None, item_map=None):
    # 전체 데이터에서 사용자와 아이템에 대한 고유 인덱스를 얻기 위한 처리
      if user_map is None or item_map is None:
        # train_data와 val_data를 합쳐서 고유한 user_idx, item_idx 매핑을 만듦
        full_data = pd.concat([train_data[['userId', 'id']], val_data[['userId', 'id']]])
        full_data['user_idx'] = full_data['userId'].astype('category').cat.codes
        full_data['item_idx'] = full_data['id'].astype('category').cat.codes
        user_map = dict(zip(full_data['userId'].unique(), full_data['user_idx'].unique()))
        item_map = dict(zip(full_data['id'].unique(), full_data['item_idx'].unique()))

      # 사용자와 아이템을 해당 인덱스로 변환
      data['user_idx'] = data['userId'].map(user_map)
      data['item_idx'] = data['id'].map(item_map)

      user_ids = torch.tensor(data['user_idx'].values, dtype=torch.long)
      item_ids = torch.tensor(data['item_idx'].values, dtype=torch.long)
      ratings = torch.tensor(data['rating'].values, dtype=torch.float32)
      numeric_features = torch.tensor(data[numeric_features_cols].values, dtype=torch.float32)

      num_users = len(user_map)  # 전체 사용자 수
      num_items = len(item_map)  # 전체 아이템 수

      dataset = TensorDataset(user_ids, item_ids, numeric_features, ratings)
      loader = DataLoader(dataset, batch_size=batch_size, shuffle=is_train, num_workers=num_workers)
      return loader, num_users, num_items, numeric_features.shape[1], user_map, item_map

    # 전체 사용자-아이템 매핑을 기반으로 train_loader, val_loader 준비
    train_loader, num_users, num_items, num_numeric, user_map, item_map = prepare(train_data, True, None, None)
    val_loader, _, _, _, _, _ = prepare(val_data, False, user_map, item_map)

    model = NeuralMF(num_users, num_items, num_numeric,  latent_dim, dropout, learning_rate)
    pruning_callback = PyTorchLightningPruningCallback(trial, monitor="validation_loss")
    trainer = pl.Trainer(
        max_epochs=epochs,
        enable_checkpointing=False, # 체크포인트 저장 비활성화
        enable_progress_bar=False, # 진행 바 비활성화
        logger=False, # 로그 저장 비활성화
        max_time="0:01:00:00",  # 최대 실행 시간 59분 설정
        accelerator="gpu"
    )
    trainer.callbacks.append(pruning_callback)  # 🔥 여기서 직접 추가
    trainer.fit(model, train_loader, val_loader)
    validation_loss = trainer.callback_metrics.get("validation_loss", torch.tensor(float('inf'))).item()
    gc.collect()  # 🔥 가비지 컬렉션 실행
    torch.cuda.empty_cache()  # 🔥 GPU 캐시 정리 (GPU 사용 시)
    return validation_loss

# ✅ Optuna 실행
pruner = optuna.pruners.MedianPruner(n_warmup_steps=5)
study = optuna.create_study(
        study_name="newMF_study",
        direction="minimize",
        storage="sqlite:///optuna_results.db",
        load_if_exists=True,
        pruner= pruner)  # 최소화 목표: 손실을 최소화하려고 함
study.optimize(objective, n_trials= 10)  # n_trials는 실험 횟수

# ✅ 최적 하이퍼파라미터 출력
print("Best Hyperparameters:", study.best_params)

[I 2025-03-31 05:40:50,104] Using an existing study with name 'newMF_study' instead of creating a new one.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO: You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
INFO:lightning.pytorch.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, 

Best Hyperparameters: {'latent_dim': 78, 'batch_size': 32, 'learning_rate': 0.0006438384247825191, 'epochs': 42, 'num_workers': 0, 'dropout': 0.26872782982430776}


In [29]:
print("Best Hyperparameters:", study.best_params)

Best Hyperparameters: {'latent_dim': 78, 'batch_size': 32, 'learning_rate': 0.0006438384247825191, 'epochs': 42, 'num_workers': 0, 'dropout': 0.26872782982430776}


In [30]:
print(f"Number of trials: {len(study.trials)}")
print(f"Completed trials: {[t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]}")


Number of trials: 17
Completed trials: [FrozenTrial(number=2, state=1, values=[0.5638096332550049], datetime_start=datetime.datetime(2025, 3, 31, 4, 14, 57, 161384), datetime_complete=datetime.datetime(2025, 3, 31, 4, 19, 8, 736096), params={'latent_dim': 125, 'batch_size': 64, 'learning_rate': 0.00012295986365149845, 'epochs': 29, 'num_workers': 3, 'dropout': 0.4525366631885158}, user_attrs={}, system_attrs={}, intermediate_values={0: 0.6580235958099365, 1: 0.6537876725196838, 2: 0.6457977890968323, 3: 0.644309401512146, 4: 0.6389192342758179, 5: 0.6368194222450256, 6: 0.634662389755249, 7: 0.6305585503578186, 8: 0.6298813223838806, 9: 0.6257390379905701, 10: 0.6236931681632996, 11: 0.6212592124938965, 12: 0.6179840564727783, 13: 0.6160552501678467, 14: 0.6173034310340881, 15: 0.6112032532691956, 16: 0.6094694137573242, 17: 0.6078788638114929, 18: 0.6047378182411194, 19: 0.6068456172943115, 20: 0.5979027152061462, 21: 0.5976426005363464, 22: 0.5871248245239258, 23: 0.5855923295021057,

In [31]:
# 각 trial의 손실 값 확인
for trial in study.trials:
    print(f"Trial {trial.number} {trial}")

Trial 0 FrozenTrial(number=0, state=3, values=None, datetime_start=datetime.datetime(2025, 3, 31, 4, 5, 45, 418401), datetime_complete=datetime.datetime(2025, 3, 31, 4, 14, 32, 604898), params={'latent_dim': 70, 'batch_size': 16, 'learning_rate': 0.00015977832984932387, 'epochs': 24, 'num_workers': 1, 'dropout': 0.2555000597354345}, user_attrs={}, system_attrs={}, intermediate_values={0: 0.6517452001571655, 1: 0.644858181476593, 2: 0.639046847820282, 3: 0.6390499472618103, 4: 0.6313342452049255, 5: 0.6249364614486694, 6: 0.6204571723937988, 7: 0.6153735518455505, 8: 0.607366681098938, 9: 0.6035739779472351, 10: 0.5986402630805969, 11: 0.5955343246459961, 12: 0.5860705971717834, 13: 0.5798872113227844, 14: 0.5727244019508362}, distributions={'latent_dim': IntDistribution(high=128, log=False, low=8, step=1), 'batch_size': CategoricalDistribution(choices=(16, 32, 64, 128)), 'learning_rate': FloatDistribution(high=0.001, log=True, low=0.0001, step=None), 'epochs': IntDistribution(high=50, 

In [32]:
#Best Hyperparameters: {'latent_dim': 43, 'batch_size': 32, 'learning_rate': 0.000532381383106112, 'epochs': 11, 'num_workers': 1}

#best_params = study.best_params

data = movies_metadata_exploded[target]
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)
small_train_data = train_data.sample(frac=0.2, random_state=42)
val_data, test_data = train_test_split(val_data, test_size=0.5, random_state=42)  # 검증 & 테스트 분리

#best_params =  {'latent_dim': 24, 'batch_size': 16, 'learning_rate': 0.00010468856330302876, 'epochs': 24, 'num_workers': 3,  'dropout': 0.47353989106173777}
best_params = {
    # 'latent_dim': 112,
    # 'batch_size': 128,
    # 'learning_rate': 0.00011357020401617381,
    # 'epochs': 42,
    # 'num_workers': 0,
    # 'dropout': 0.4060787393086205
    # 'latent_dim': 91,
    # 'batch_size': 16,
    # 'learning_rate': 0.0003131425208960249,
    # 'epochs': 21,
    # 'num_workers': 2,
    # 'dropout': 0.07151845254858118
    # 'latent_dim': 53,
    # 'batch_size': 16,
    # 'learning_rate': 0.0004096453948261441,
    # 'epochs': 11,
    # 'num_workers': 1,
    # 'dropout': 0.24716962709299214
    'latent_dim': 78,
    'batch_size': 32,
    'learning_rate': 0.0006438384247825191,
    'epochs': 42,
    'num_workers': 0,
    'dropout': 0.26872782982430776
}
latent_dim = best_params['latent_dim']
batch_size = best_params['batch_size']
learning_rate = best_params['learning_rate']
epochs = best_params['epochs']
num_workers = best_params['num_workers']
dropout = best_params['dropout']



def prepare(data, is_train, user_map=None, item_map=None):
    # 전체 데이터에서 사용자와 아이템에 대한 고유 인덱스를 얻기 위한 처리
    if user_map is None or item_map is None:
        # train_data와 val_data를 합쳐서 고유한 user_idx, item_idx 매핑을 만듦
        full_data = pd.concat([train_data[['userId', 'id']], val_data[['userId', 'id']]])
        full_data['user_idx'] = full_data['userId'].astype('category').cat.codes
        full_data['item_idx'] = full_data['id'].astype('category').cat.codes
        user_map = dict(zip(full_data['userId'].unique(), full_data['user_idx'].unique()))
        item_map = dict(zip(full_data['id'].unique(), full_data['item_idx'].unique()))

    # 사용자와 아이템을 해당 인덱스로 변환
    data['user_idx'] = data['userId'].map(user_map)
    data['item_idx'] = data['id'].map(item_map)

    user_ids = torch.tensor(data['user_idx'].values, dtype=torch.long)
    item_ids = torch.tensor(data['item_idx'].values, dtype=torch.long)
    ratings = torch.tensor(data['rating'].values, dtype=torch.float32)
    numeric_features = torch.tensor(data[numeric_features_cols].values, dtype=torch.float32)

    num_users = len(user_map)  # 전체 사용자 수
    num_items = len(item_map)  # 전체 아이템 수

    dataset = TensorDataset(user_ids, item_ids, numeric_features, ratings)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=is_train, num_workers=num_workers)
    return loader, num_users, num_items, numeric_features.shape[1], user_map, item_map

# 전체 사용자-아이템 매핑을 기반으로 train_loader, val_loader 준비
train_loader, num_users, num_items, num_numeric, user_map, item_map = prepare(train_data, True, None, None)
val_loader, _, _, _, _, _ = prepare(val_data, False, user_map, item_map)


model = NeuralMF(num_users, num_items, num_numeric, latent_dim, dropout, learning_rate)
trainer = pl.Trainer(
    max_epochs=epochs,
    enable_checkpointing=False, # 체크포인트 저장 비활성화
    enable_progress_bar=False, # 진행 바 비활성화
    logger=False # 로그 저장 비활성화
)
trainer.fit(model, train_loader, val_loader)

# 모델 학습이 끝난 후 불필요한 변수 제거
#del train_loader, val_loader
torch.cuda.empty_cache()  # GPU 메모리 해제
gc.collect()  # Python 가비지 컬렉션 실행



INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name           | Type      | Params | Mode 
-----------------------------------------------------
0 | user_embedding | Embedding | 52.3 K | train
1 | item_embedding | Embedding | 218 K  | train
2 | numeric_fc     | Linear    | 702    | train
3 | gmf_fc         | Linear    | 6.2 K  | train
4 | mlp_fc1        | Linear    | 30.1 K | train
5 | mlp_fc2        | Linear    | 8.3 K  | train
6 | final_fc       | Linear    | 143    | train
7 | dropout        | Dropout   | 0      | train
8 | 

1688

In [33]:
torch.save(model.state_dict(), "best_model.pth")

In [34]:
num_users, num_items

(671, 2807)

In [36]:
full_data = pd.concat([train_data[['userId', 'id']], val_data[['userId', 'id']]])
full_data['user_idx'] = full_data['userId'].astype('category').cat.codes
full_data['item_idx'] = full_data['id'].astype('category').cat.codes
user_map = dict(zip(full_data['userId'].unique(), full_data['user_idx'].unique()))
item_map = dict(zip(full_data['id'].unique(), full_data['item_idx'].unique()))


num_users =len(user_map)
num_items = len(item_map)
numeric_features = torch.tensor(train_data[numeric_features_cols].values, dtype=torch.float32)
num_numeric = numeric_features.shape[1]
# latent_dim =
# learning_rate = 0.00011357020401617381,
# dropout = 0.4060787393086205

#   'latent_dim': 53,
#     'batch_size': 16,
#     'learning_rate': 0.0004096453948261441,
#     'epochs': 11,
#     'num_workers': 1,
#     'dropout': 0.24716962709299214
model = NeuralMF(num_users, num_items, num_numeric, latent_dim, dropout, learning_rate)

# # 저장된 모델 로드
model.load_state_dict(torch.load('best_model.pth'))  # 모델을 불러올 경로
model.eval()  # 평가 모드로 변경

<ipython-input-36-5b87112d70a8>:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pth'))  # 모델을 불러올 경로


NeuralMF(
  (user_embedding): Embedding(671, 78, max_norm=1.0)
  (item_embedding): Embedding(2807, 78, max_norm=1.0)
  (numeric_fc): Linear(in_features=8, out_features=78, bias=True)
  (gmf_fc): Linear(in_features=78, out_features=78, bias=True)
  (mlp_fc1): Linear(in_features=234, out_features=128, bias=True)
  (mlp_fc2): Linear(in_features=128, out_features=64, bias=True)
  (final_fc): Linear(in_features=142, out_features=1, bias=True)
  (dropout): Dropout(p=0.26872782982430776, inplace=False)
  (activation): ReLU()
  (criterion): MSELoss()
)

In [39]:
#test_loader, _, _, _ = prepare(test_data, False)
test_loader, _, _, _, _, _ = prepare(test_data, False, user_map, item_map)
predictions = trainer.predict(model, test_loader)


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


In [40]:
for user_ids, item_ids, numeric_features, ratings in test_loader:
  print(user_ids.shape)
  print(item_ids.shape)
  print(numeric_features.shape)
  print(ratings.shape)
  break

torch.Size([32])
torch.Size([32])
torch.Size([32, 8])
torch.Size([32])


In [41]:
predicted_ratings = []
actual_ratings = []
for pred, rating in predictions:
    predicted_ratings.append(pred)
    actual_ratings.append(rating)

In [42]:
# ✅ MSE 계산 (sklearn)
mse = mean_squared_error(actual_ratings, predicted_ratings)
print(f"Test MSE: {mse}")

Test MSE: 0.42061888611859344


In [43]:
mae = mean_absolute_error(actual_ratings, predicted_ratings)
print(f"Test MAE: {mae}")

Test MAE: 0.4876791064053514


In [44]:
#RMSE 계산
rmse = np.sqrt(mse)
print(f"Test RMSE: {rmse}")

Test RMSE: 0.648551375080335


In [45]:
torch.save(model.state_dict(), "best_model.pth")

In [46]:
#train_data.head(),
item_title = "장화, 홍련"
int(movies_metadata_exploded[movies_metadata_exploded['original_title'] == item_title]['id'].iloc[0])

4552

In [47]:
# 유저 1의 user_idx 찾기
user_id = 2
item_title = "장화, 홍련"
item_id = int(movies_metadata_exploded[movies_metadata_exploded['original_title'] == item_title]['id'].iloc[0])

user_idx = train_data[train_data['userId'] == user_id]['user_idx'].unique()[0]
item_idx = train_data[train_data['id'] == item_id]['item_idx'].unique()[0]

print(f"유저 {user_id} user_index: {user_idx}, {item_id} item_index: {item_idx}")

numeric_item_features = train_data[train_data['item_idx'] == item_idx][numeric_features_cols]

numeric_item_features.mean().values


유저 2 user_index: 1, 4552 item_index: 1605


array([ 0.14000068,  0.22755207, -0.17710736,  0.42516245,  1.0531892 ,
       -0.9873358 ,  0.53024706, -0.20964886])

In [48]:
user_ids = torch.tensor([user_idx], dtype=torch.long)  # 예: 사용자 아이디
item_ids = torch.tensor([item_idx], dtype=torch.long)  # 예: 아이템 아이디
numeric_features = torch.tensor(numeric_item_features.mean().values,  dtype=torch.float32)  # 예: 숫자형 특성 (예: 나이, 성별, 등)
item_features = numeric_features.reshape(1, -1)
#user_ids.shape, item_ids.shape,numeric_features.reshape(1, -1).shape
# 모델을 통해 예측 수행
predicted_rating = model(user_ids, item_ids, item_features)
# 예측된 평점 출력
print(predicted_rating.item())

4.097311496734619


In [49]:
import torch

def batch_recommend(model, user_ids, candidate_items, numeric_features_df, K):
    """
    배치 방식으로 추천 아이템을 계산하는 함수

    Args:
        model: 학습된 NeuralMF 모델
        user_ids: 사용자 ID 목록 (배치 처리)
        candidate_items: 추천 후보 아이템 리스트
        numeric_features_df: 사용자-아이템 별 numeric_features (Pandas DataFrame)
        K: 추천할 아이템 개수

    Returns:
        recommendations: {user_id: [top-K 추천 아이템]}
    """
    model.eval()  # 평가 모드로 변경 (드롭아웃 등 비활성화)

    # ✅ (1) 유니크한 사용자 ID와 아이템 ID를 가져옴
    user_ids = torch.tensor(user_ids.unique(), dtype=torch.long)  # 사용자 ID 텐서 변환
    item_ids = torch.tensor(candidate_items.unique(), dtype=torch.long)  # 아이템 ID 텐서 변환

    # ✅ (2) numeric_features_df의 인덱스 설정
    numeric_features_df = numeric_features_df.set_index(['user_idx', 'item_idx']).sort_index()  # ✅ 컬럼명 수정

    # ✅ (3) 각 사용자-아이템 쌍에 대해 numeric_features를 가져옴
    numeric_features_list = []
    for user_id in user_ids:
        for item_id in item_ids:
            user_int = user_id.item()  # 텐서를 Python int로 변환
            item_int = item_id.item()  # 텐서를 Python int로 변환

            # numeric_features 가져오기
            if (user_int, item_int) in numeric_features_df.index:
                feature_values = numeric_features_df.loc[(user_int, item_int)].values[0]
            else:
                # 없으면 기본값 (0 벡터)
                 feature_values = np.zeros(numeric_features_df.shape[1])  # 기본값은 0 벡터

            numeric_features_list.append(feature_values)

    # ✅ (4) numeric_features를 PyTorch 텐서로 변환
    numeric_features = torch.tensor(np.array(numeric_features_list), dtype=torch.float32)

    # ✅ (5) 모든 사용자-아이템 조합 생성
    user_batch = user_ids.repeat_interleave(len(item_ids))  # (사용자 개수 * 아이템 개수)
    item_batch = item_ids.repeat(len(user_ids))  # (사용자 개수 * 아이템 개수)

    # ✅ (6) 배치 예측 실행
    with torch.no_grad():
        scores = model(user_batch, item_batch, numeric_features)  # (사용자*아이템 개수)

    # ✅ (7) 각 사용자별로 Top-K 아이템 추천
    scores = scores.view(len(user_ids), len(item_ids))  # (사용자 개수, 아이템 개수)
    top_k_indices = torch.argsort(scores, dim=1, descending=True)[:, :K]  # 상위 K개 인덱스

    # ✅ (8) 추천 결과를 딕셔너리 형태로 변환
    recommendations = {user_ids[i].item(): [item_ids[idx].item() for idx in top_k_indices[i]]
                       for i in range(len(user_ids))}
    return recommendations

# ✅ 테스트 실행


K = 10
user_ids = train_data['user_idx']
candidate_items = train_data['item_idx']
numeric_features_cols = ['user_mean_rating', 'user_rating_std', 'user_review_count',
       'release_year', 'movie_mean_rating', 'movie_review_count', 'genre_ids',
       'user_preference']
numeric_features_cols.append('user_idx')
numeric_features_cols.append('item_idx')
numeric_features_df = train_data[numeric_features_cols]

recommendations = batch_recommend(model, user_ids, candidate_items, numeric_features_df, K)

In [50]:
len(recommendations)

671

In [51]:
def hit_rate(recommendations, test_data):
    """
    HitRate 계산 (HR@K)

    Args:
        recommendations: {user_id: [추천 아이템 리스트]}
        test_data: 테스트 데이터 (실제 사용자-아이템 매칭, DataFrame)

    Returns:
        hit_rate: HR@K 값
    """
    hits = 0
    total_users = len(recommendations)

    for user_idx, recommended_items in recommendations.items():
        actual_items = set(test_data[test_data['user_idx'] == user_idx]['item_idx'])  # 실제 선호 아이템
        if any(item in actual_items for item in recommended_items):  # 적어도 1개 포함되면 hit
            hits += 1

    return hits / total_users  # HitRate 계산

hit_rate(recommendations, test_data)

0.6959761549925484

In [52]:
def precision_at_k(recommendations, test_data, K):
    """
    Precision@K 계산

    Args:
        recommendations: {user_id: [추천 아이템 리스트]}
        test_data: 테스트 데이터 (실제 사용자-아이템 매칭, DataFrame)
        K: 추천 아이템 개수

    Returns:
        precision_k: Precision@K 값
    """
    total_precision = 0
    total_users = len(recommendations)

    for user_idx, recommended_items in recommendations.items():
        actual_items = set(test_data[test_data['user_idx'] == user_idx]['item_idx'])  # 실제 선호 아이템
        num_hits = sum(1 for item in recommended_items[:K] if item in actual_items)  # K개 중 실제 선호한 아이템 개수

        precision = num_hits / K  # Precision 계산
        total_precision += precision

    return total_precision / total_users  # 평균 Precision@K

In [53]:
K = 10  # 추천 리스트 길이
precision_k = precision_at_k(recommendations, test_data, K)
print(f"Precision@{K}: {precision_k:.4f}")


Precision@10: 0.1396


In [54]:
import numpy as np

def dcg_at_k(relevance, k):
    """
    DCG 계산 함수
    """
    return np.sum(relevance[:k] / np.log2(np.arange(2, k+2)))

def ndcg_at_k(recommendations, test_data, k):
    """
    NDCG@K 계산 함수
    Args:
        recommendations: {user_id: [추천 아이템 리스트]}
        test_data: 테스트 데이터 (실제 사용자-아이템 매칭, DataFrame)
        k: 추천 아이템의 개수

    Returns:
        NDCG@K 값
    """
    ndcg_total = 0
    total_users = len(recommendations)

    for user_idx, recommended_items in recommendations.items():
        # 사용자별 실제 평점 또는 관련성 점수를 가져옵니다.
        actual_items = test_data[test_data['user_idx'] == user_idx]

        # 추천 아이템의 실제 관련성 점수(평점)
        relevance = []
        for item in recommended_items[:k]:
            # 실제 평점 또는 아이템의 관련성 점수를 가져옴
            if item in actual_items['item_idx'].values:
                # 실제 평점은 해당 항목에서 가져옵니다
                relevance.append(actual_items[actual_items['item_idx'] == item]['rating'].values[0])
            else:
                relevance.append(0)  # 평점이 없으면 관련성 점수는 0

        # DCG 계산
        dcg = dcg_at_k(np.array(relevance), k)

        # 이상적인 DCG 계산 (이상적인 순서는 평점이 높은 순서)
        ideal_relevance = sorted(relevance, reverse=True)
        idcg = dcg_at_k(np.array(ideal_relevance), k)

        # NDCG 계산
        if idcg == 0:
            ndcg_total += 0  # idcg가 0일 때는 NDCG도 0
        else:
            ndcg_total += dcg / idcg

    # 평균 NDCG
    return ndcg_total / total_users

# 예시 사용법
# test_data는 'user_idx', 'item_idx', 'rating' 열을 포함해야 합니다.
# recommendations는 {user_id: [추천된 아이템 리스트]} 형식이어야 합니다.
ndcg = ndcg_at_k(recommendations, test_data, k=10)
print("NDCG@10:", ndcg)


NDCG@10: 0.4101096956714116
